In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [2]:
df = pd.read_csv('data/stud.csv')

In [3]:
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [4]:
X = df.drop(columns=['math_score'])
y = df['math_score']

In [5]:
X.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [6]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math_score, Length: 1000, dtype: int64

In [7]:
# Create  column transformer With 3 types of transformers
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OnehotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

/var/folders/w6/4s__vlwd3qd6fb66nzf4ts6r0000gn/T/ipykernel_79019/4066756903.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include="object").columns


In [8]:
X = preprocessor.fit_transform(X)

In [9]:
X

array([[ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.19399858,  0.39149181],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         1.42747598,  1.31326868],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.77010859,  1.64247471],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.12547206, -0.20107904],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.60515772,  0.58901542],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.15336989,  1.18158627]], shape=(1000, 19))

In [10]:
X.shape

(1000, 19)

In [11]:
## Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
X_train.shape, X_test.shape

((800, 19), (200, 19))

## Create and Evaluate function to print all metrics after model training

In [13]:
def evaluate_model(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    r2_score_value = r2_score(actual, predicted)
    return mae, rmse, r2_score_value

In [15]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbours Regression": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random forest Regressor": RandomForestRegressor(),
    "XGBoostRegressor": XGBRegressor(),
    "CatBoosting Regressor": CatBoostRegressor(),
    "Adaboost Regressor": AdaBoostRegressor()
}

model_list = []
r2_list = []


for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) ## Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate the train and test dataset
    model_train_mae, model_train_rmse, mode_train_r2_score = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2_score = evaluate_model(y_test, y_test_pred)

    model_name = list(models.keys())[i]
    print(model_name)

    model_list.append(model_name)

    print("Model performance for training set")
    print("- Root mean squared error: {:.4f}".format(model_train_mae))
    print("- Mean absolute error: {:.4f}".format(model_train_rmse))
    print("- R2 score: {:.4f}".format(mode_train_r2_score))

    print("----------------------")

    print("Model performace for test set")
    print("- Root mean squared error: {:.4f}".format(model_test_mae))
    print("- Mean absolute error: {:.4f}".format(model_test_rmse))
    print("- R2 score: {:.4f}".format(model_test_r2_score))
    r2_list.append(model_test_r2_score)

    print('='*35)
    print('\n')
    



Linear Regression
Model performance for training set
- Root mean squared error: 4.2667
- Mean absolute error: 5.3231
- R2 score: 0.8743
----------------------
Model performace for test set
- Root mean squared error: 4.2148
- Mean absolute error: 5.3940
- R2 score: 0.8804


Lasso
Model performance for training set
- Root mean squared error: 5.2063
- Mean absolute error: 6.5938
- R2 score: 0.8071
----------------------
Model performace for test set
- Root mean squared error: 5.1579
- Mean absolute error: 6.5197
- R2 score: 0.8253


Ridge
Model performance for training set
- Root mean squared error: 4.2650
- Mean absolute error: 5.3233
- R2 score: 0.8743
----------------------
Model performace for test set
- Root mean squared error: 4.2111
- Mean absolute error: 5.3904
- R2 score: 0.8806


K-Neighbours Regression
Model performance for training set
- Root mean squared error: 4.5270
- Mean absolute error: 5.7172
- R2 score: 0.8550
----------------------
Model performace for test set
- Root 